In [0]:
# ==============================================================
# Master Orchestrator – Superstore Medallion Lakehouse
# Enterprise-Ready Version with Full Traceability
# Author: Biresh Tamang
# Purpose:
#   - Generate a single master_run_id for the entire pipeline
#   - Execute all layer notebooks sequentially (Bronze → Silver → Gold → Marts → Metrics → Features → Dashboards)
#   - Capture and log child layer_run_ids for full traceability
#   - Log total pipeline runtime and status
#   - Designed for production deployment in Delta Lake Medallion Architecture
# ==============================================================

import time
from datetime import datetime
import sys

# -----------------------------
# Add shared utilities path
# -----------------------------
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_logger import get_superstore_logger, log_event, create_master_run_id

# -----------------------------
# Initialize Master Run ID & Logger
# -----------------------------
master_run_id = create_master_run_id()  # Unique ID for this full pipeline run
logger_master = get_superstore_logger("superstore_master_orchestrator")

# Log master pipeline initiation
log_event(
    logger_master,
    "INFO",
    "Initiating Superstore Master Pipeline using Delta Lake Medallion Architecture",
    master_run_id=master_run_id
)

pipeline_start = time.time()  # Start timer for pipeline runtime

# -----------------------------
# Notebook Paths (Layer Orchestrators)
# -----------------------------
bronze_layer_etl_orchestrator_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/superstore_orchestrator/layer_orchestrator/superstore_bronze_layer_ETL_orchestrator"
silver_layer_etl_orchestrator_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/superstore_orchestrator/layer_orchestrator/superstore_silver_layer_ETL_orchestrator"
gold_layer_dim_etl_orchestrator_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/superstore_orchestrator/layer_orchestrator/superstore_gold_dimensional_orchestrator"
gold_layer_fact_etl_orchestrator_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/superstore_orchestrator/layer_orchestrator/superstore_gold_facts_orchestrator"

marts_paths = [
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/marts/mart_customer_360",
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/marts/mart_production_performance",
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/marts/mart_sales_daily"
]

metrics_paths = [
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/metrics/metrics_business_kpi",
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/metrics/metrics_daily_kpi"
]

features_paths = [
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/features/features_customer",
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/features/features_product",
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/features/features_sales_timeseries"
]

dashboards_paths = [
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/dashboards/01_superstore_executive_overview",
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/dashboards/02_superstore_customer_analytics",
    "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/dashboards/03_superstore_product_performance"
]



# -----------------------------
# Helper Function: Run Notebook with Logging & Layer ID Capture
# -----------------------------
def run_notebook(path, retries=3, timeout_seconds=1800, retry_delay=60):
    """
    Executes a child notebook with retries, timeout, and logging.

    Args:
        path (str): Notebook path to run.
        retries (int): Number of retry attempts on failure.
        timeout_seconds (int): Timeout for notebook execution.
        retry_delay (int): Seconds to wait between retries.

    Returns:
        str: layer_run_id of the executed notebook.
    """
    attempt = 0
    while attempt <= retries:
        attempt += 1
        try:
            log_event(
                logger_master,
                "INFO",
                f"Running notebook: {path} | Attempt {attempt}/{retries+1}",
                master_run_id=master_run_id
            )

            # Run the notebook with timeout
            layer_run_id = dbutils.notebook.run(
                path,
                timeout_seconds=timeout_seconds,
                arguments={"master_run_id": master_run_id}
            )

            log_event(
                logger_master,
                "INFO",
                f"Notebook completed successfully: {path}",
                master_run_id=master_run_id,
                layer_run_id=layer_run_id
            )

            return layer_run_id

        except Exception as e:
            log_event(
                logger_master,
                "WARN",
                f"Attempt {attempt} failed for notebook {path} | Error: {str(e)}",
                master_run_id=master_run_id
            )
            if attempt > retries:
                log_event(
                    logger_master,
                    "ERROR",
                    f"Notebook failed after {retries+1} attempts: {path}",
                    master_run_id=master_run_id
                )
                raise
            time.sleep(retry_delay)

# -----------------------------
# Sequentially Run Layers
# -----------------------------
layer_ids = []  # Keep track of all layer_run_ids for final logging

# Core Layers
layer_ids.append(run_notebook(bronze_layer_etl_orchestrator_path))
layer_ids.append(run_notebook(silver_layer_etl_orchestrator_path))
layer_ids.append(run_notebook(gold_layer_dim_etl_orchestrator_path))
layer_ids.append(run_notebook(gold_layer_fact_etl_orchestrator_path))

# Marts
for path in marts_paths:
    layer_ids.append(run_notebook(path))

# Metrics
for path in metrics_paths:
    layer_ids.append(run_notebook(path))

# Features
for path in features_paths:
    layer_ids.append(run_notebook(path))

# Dashboards
for path in dashboards_paths:
    layer_ids.append(run_notebook(path))


# -----------------------------
# End Master Pipeline
# -----------------------------
pipeline_end = time.time()
total_seconds = round(pipeline_end - pipeline_start, 2)

# Log master completion with all child layer_run_ids
log_event(
    logger_master,
    "INFO",
    "Master pipeline completed successfully",
    master_run_id=master_run_id,
    layer_run_id=str(layer_ids),  # include all child layer IDs for traceability
    total_runtime_seconds=total_seconds
)

# Exit notebook with final status
dbutils.notebook.exit(
    f"MASTER PIPELINE SUCCESS | master_run_id={master_run_id} | runtime={total_seconds}s"
)